In [1]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

pd.set_option("display.max_columns", None)

In [2]:
PROJECT_ROOT = Path.cwd().parent

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

print(PROJECT_ROOT)
print(DATA_PROCESSED)

C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation
C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation\data\processed


In [3]:
model_df = pd.read_parquet(
    DATA_PROCESSED / "modeling_features.parquet"
)

print(model_df.shape)

model_df.head()

(93358, 11)


,recency_days,frequency,monetary_value,total_items,freight_to_merchandise_ratio,avg_installments,avg_review_score,review_coverage_ratio,unique_products,unique_categories,unique_sellers
0,111,1,129.90,1,0.092379,8.0,5.0,1.0,1,1,1
1,114,1,18.90,1,0.438624,1.0,4.0,1.0,1,1,1
2,537,1,69.00,1,0.249565,8.0,3.0,1.0,1,1,1
3,321,1,25.99,1,0.678338,4.0,4.0,1.0,1,1,1
4,288,1,180.00,1,0.093833,6.0,5.0,1.0,1,1,1


In [4]:
validation = pd.Series({
    "rows": len(model_df),
    "columns": len(model_df.columns),
    "missing_values": model_df.isna().sum().sum(),
    "duplicate_rows": model_df.duplicated().sum()
})

validation

rows              93358
columns              11
missing_values      604
duplicate_rows      792
dtype: int64

In [5]:
model_df.isna().sum().sort_values(ascending=False)

avg_review_score                603
avg_installments                  1
recency_days                      0
frequency                         0
monetary_value                    0
total_items                       0
freight_to_merchandise_ratio      0
review_coverage_ratio             0
unique_products                   0
unique_categories                 0
unique_sellers                    0
dtype: int64

In [6]:
# Imputation


imputer = SimpleImputer(strategy="median")

model_imputed = pd.DataFrame(
    imputer.fit_transform(model_df),
    columns=model_df.columns
)

model_imputed.head()

,recency_days,frequency,monetary_value,total_items,freight_to_merchandise_ratio,avg_installments,avg_review_score,review_coverage_ratio,unique_products,unique_categories,unique_sellers
0,111.0,1.0,129.90,1.0,0.092379,8.0,5.0,1.0,1.0,1.0,1.0
1,114.0,1.0,18.90,1.0,0.438624,1.0,4.0,1.0,1.0,1.0,1.0
2,537.0,1.0,69.00,1.0,0.249565,8.0,3.0,1.0,1.0,1.0,1.0
3,321.0,1.0,25.99,1.0,0.678338,4.0,4.0,1.0,1.0,1.0,1.0
4,288.0,1.0,180.00,1.0,0.093833,6.0,5.0,1.0,1.0,1.0,1.0


In [7]:
# Validate Imputation

pd.Series({
    "rows": len(model_imputed),
    "columns": len(model_imputed.columns),
    "missing_values": model_imputed.isna().sum().sum()
})

rows              93358
columns              11
missing_values        0
dtype: int64

In [8]:
scaler = RobustScaler()

scaled_array = scaler.fit_transform(model_imputed)

scaled_df = pd.DataFrame(
    scaled_array,
    columns=model_imputed.columns
)

scaled_df.head()

,recency_days,frequency,monetary_value,total_items,freight_to_merchandise_ratio,avg_installments,avg_review_score,review_coverage_ratio,unique_products,unique_categories,unique_sellers
0,-0.461207,0.0,0.375114,0.0,-0.535578,2.000000,0.0,0.0,0.0,0.0,0.0
1,-0.448276,0.0,-0.661422,0.0,0.870328,-0.333333,-1.0,0.0,0.0,0.0,0.0
2,1.375000,0.0,-0.193580,0.0,0.102667,2.000000,-2.0,0.0,0.0,0.0,0.0
3,0.443966,0.0,-0.595214,0.0,1.843668,0.666667,-1.0,0.0,0.0,0.0,0.0
4,0.301724,0.0,0.842956,0.0,-0.529672,1.333333,0.0,0.0,0.0,0.0,0.0


In [9]:
# Validation


pd.Series({
    "rows": len(scaled_df),
    "columns": len(scaled_df.columns),
    "missing_values": scaled_df.isna().sum().sum()
})

rows              93358
columns              11
missing_values        0
dtype: int64

In [10]:
# Save the Scaled Dataset

SCALED_DATA_PATH = DATA_PROCESSED / "scaled_features.parquet"

scaled_df.to_parquet(
    SCALED_DATA_PATH,
    index=False
)

print("Saved:", SCALED_DATA_PATH)

Saved: C:\Users\amolm\Desktop\PROJECTS RESUME\Customer Segmentation\data\processed\scaled_features.parquet


In [11]:
# Reload & Validate

scaled_test = pd.read_parquet(
    SCALED_DATA_PATH
)

phase5_validation = pd.Series({
    "rows": len(scaled_test),
    "columns": len(scaled_test.columns),
    "missing_values": scaled_test.isna().sum().sum(),
    "duplicate_rows": scaled_test.duplicated().sum()
})

phase5_validation

rows              93358
columns              11
missing_values        0
duplicate_rows      792
dtype: int64